# Phase 5 - Logistic Regression: is this a high-demand hour?

**Team IDs:** 16002932, 16004208, 16006616  |  **Seed:** 42648

**Rules:** classifier trained with our own gradient descent (`gradient_descent.gd_fit`); no sklearn `.fit()` / `.predict()` / `LogisticRegression`; sklearn metrics appear only inside tests; `test.csv` is never used here.

## Expectation

In [1]:
# -- Imports ------------------------------------------------------------------
import warnings, hashlib, json, sys, time, re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split          # seeded split only
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score   # TESTS ONLY (cross-check of our numpy metrics)

import gradient_descent as core
import polynomial_regression as p2
import bias_variance as p3
import regularization as p4
import logistic_regression as lg

warnings.filterwarnings('ignore')
np.set_printoptions(precision=6, suppress=True)

SEED = 42648
IDS = ['16002932', '16006616', '16004208']
DATA_DIR = Path('Datasets')
P1_STEM, P2_STEM, P3_STEM, P4_STEM = ('gradient_descent_artifacts', 'polynomial_regression_artifacts',
                                      'bias_variance_artifacts', 'regularization_artifacts')
P5_STEM = 'logistic_regression_artifacts'
PLOTS = Path('plots'); PLOTS.mkdir(exist_ok=True)
T0 = time.time()
print(f'Python {sys.version.split()[0]}  numpy {np.__version__}  pandas {pd.__version__}')

Python 3.14.6  numpy 2.5.0  pandas 3.0.3


## 1. Seed and chain verification (Req 1-2)

In [2]:
# Seed and seeded split
assert core.compute_seed(IDS) == SEED, 'seed mismatch'
train_all = pd.read_csv(DATA_DIR / 'train.csv')
train_df, val_df = train_test_split(train_all, test_size=0.20, random_state=SEED)

_, _, meta_p1, train_idx, val_idx = core.load_artifacts(P1_STEM)
assert np.array_equal(train_df.index.values, train_idx) and np.array_equal(val_df.index.values, val_idx)
print(f'[PASS] seed {SEED}; split train={len(train_df)}, val={len(val_df)} matches the stored indices')

# Phase 4 artifacts: hash, number of non-zeros, feature list
z4 = np.load(P4_STEM + '.npz', allow_pickle=False)
with open(P4_STEM + '.json', encoding='utf-8') as f:
    meta_p4 = json.load(f)
sha_l1 = hashlib.sha256(z4['l1_w'].tobytes()).hexdigest()
assert sha_l1 == meta_p4['phase4_l1_w_sha256'], 'Phase 4 L1 weight hash mismatch'
FEATS = meta_p4['l1_nonzero_features']
n_nonzero = int((np.abs(z4['l1_w']) > 1e-8).sum())
assert n_nonzero == len(FEATS) > 0, (n_nonzero, len(FEATS))
assert FEATS == [n for n, w in zip(meta_p4['feature_names'], z4['l1_w']) if abs(w) > 1e-8]
print(f'[PASS] Phase 4 L1 hash matches; {n_nonzero} non-zero Lasso weights = {len(FEATS)} listed features')

# Phase 3 artifacts: hash chain
w_p3, b_p3, meta_p3 = p2.load_artifacts(P3_STEM)
assert hashlib.sha256(w_p3.tobytes()).hexdigest() == meta_p3['phase3_w_sha256'] == meta_p4['phase3_w_sha256']
print('[PASS] Phase 3 hash chain (Phase 3 artifact == hash recorded in Phase 4)')

poly_params = {'term_specs': meta_p3['term_specs'], 'poly_means': meta_p3['poly_means'],
               'poly_stds': meta_p3['poly_stds'], 'n_new_cols': meta_p3['n_new_cols']}

[PASS] seed 42648; split train=8708, val=2178 matches the stored indices
[PASS] Phase 4 L1 hash matches; 470 non-zero Lasso weights = 470 listed features
[PASS] Phase 3 hash chain (Phase 3 artifact == hash recorded in Phase 4)


## 2. Feature matrix: Phase 1 preprocessing, Phase 3 expansion, Phase 4 Lasso subset (Req 3)

In [3]:
X_tr, FEAT_NAMES = lg.build_features(train_df, meta_p1, poly_params, FEATS)
X_vl, _          = lg.build_features(val_df,   meta_p1, poly_params, FEATS)
print('X_train', X_tr.shape, ' X_val', X_vl.shape)
assert X_tr.shape == (8708, len(FEATS)) and X_vl.shape == (2178, len(FEATS))
assert np.isfinite(X_tr).all() and np.isfinite(X_vl).all()
assert not set(FEAT_NAMES) & {'cnt', 'instant', 'dteday', 'high'}
assert FEAT_NAMES == FEATS
print('[PASS] shapes, finiteness, no forbidden columns, column order = Phase 4 list')

# Chain check: the full (pre-selection) matrix with Phase 4's L1 weights reproduces Phase 4's stored validation R2
X_p4_vl, _ = lg.build_features(val_df, meta_p1, poly_params, meta_p4['feature_names'])
r2_chain = core.r2_score_np(val_df['cnt'].values.astype(float), core.predict(X_p4_vl, z4['l1_w'], float(z4['l1_b'][0])))
assert abs(r2_chain - meta_p4['l1_r2_val']) < 1e-9, (r2_chain, meta_p4['l1_r2_val'])
print(f'[PASS] Phase 4 Lasso val R2 reproduced from artifacts: {r2_chain:.6f}')

X_train (8708, 470)  X_val (2178, 470)
[PASS] shapes, finiteness, no forbidden columns, column order = Phase 4 list


[PASS] Phase 4 Lasso val R2 reproduced from artifacts: 0.907946


## 3. Target: "high demand" relative to comparable hours (Req 4)

`high = 1` if `cnt` is strictly greater than the **training** 75th percentile of `cnt` within the same (`hr`, `workingday`) group (48 groups). Thresholds are computed from training rows only and then applied unchanged to validation.

In [4]:
THR = lg.group_thresholds(train_df)                 # raises ValueError if any of the 48 groups is empty
small = lg.small_groups(train_df, min_n=5)
print('groups:', len(THR), '| groups with < 5 train rows:', small)
y_tr = lg.make_target(train_df, THR)
y_vl = lg.make_target(val_df,   THR)
assert 0.20 <= y_tr.mean() <= 0.30
print(f'positive rate  train {y_tr.mean():.4f}   val {y_vl.mean():.4f}')

tmp = train_df.assign(high=y_tr).groupby(['workingday', 'hr'])['high'].mean().unstack(0)
print('\nTrain positive rate per group: min {:.3f}  max {:.3f}'.format(tmp.values.min(), tmp.values.max()))
tmp_v = val_df.assign(high=y_vl).groupby(['workingday', 'hr'])['high'].mean().unstack(0)
print('Val   positive rate per group: min {:.3f}  max {:.3f}'.format(tmp_v.values.min(), tmp_v.values.max()))
print('\nExample thresholds (cnt):  hr=8 workday {:.0f}, hr=8 non-workday {:.0f}, hr=3 workday {:.0f}, hr=13 non-workday {:.0f}'.format(
    THR[(8, 1)], THR[(8, 0)], THR[(3, 1)], THR[(13, 0)]))

groups: 48 | groups with < 5 train rows: {}
positive rate  train 0.2441   val 0.2502

Train positive rate per group: min 0.168  max 0.255
Val   positive rate per group: min 0.077  max 0.438

Example thresholds (cnt):  hr=8 workday 651, hr=8 non-workday 144, hr=3 workday 7, hr=13 non-workday 492


## 4. Why one global threshold is flawed (Req 5)

We build the naive target (train 75th percentile of `cnt`, one value for every hour) and fit an **hour-only** logistic model (the 23 hour dummies, nothing else) to both targets. If a target can be predicted from the clock alone, a classifier trained on it learns the clock, not weather or season.

In [5]:
# -- Shared fitting helper (our own GD; sklearn is not used to fit) ------------------
FIT_KW = dict(max_iter=300000, tol=1e-8, gtol=1e-7)

def fit_logreg(X, y, l2=0.0, sw=None, w0=None, b0=0.0, X_val=None, y_val=None, record_every=100, **kw):
    grad_fn, loss_fn = lg.make_fns(l2, sw)
    L = lg.lipschitz_logistic(X, l2, sw)
    res = core.gd_fit(X, y, lr=1.0 / L, w0=w0, b0=b0, grad_fn=grad_fn, loss_fn=loss_fn, momentum=True,
                      X_val=X_val, y_val=y_val, record_every=record_every, **{**FIT_KW, **kw})
    res['L'], res['lr'] = L, 1.0 / L
    return res

# Hour-only design: the 23 hr dummies of the Phase 1 preprocessing
X1_tr, n1 = core.transform(train_df, meta_p1)
X1_vl, _  = core.transform(val_df,   meta_p1)
hr_cols = [i for i, n in enumerate(n1) if n.startswith('hr_')]
assert len(hr_cols) == 23
H_tr, H_vl = X1_tr[:, hr_cols], X1_vl[:, hr_cols]

y_tr_g = lg.global_target(train_df, train_df)
y_vl_g = lg.global_target(train_df, val_df)
print(f'global-threshold target: train 75th percentile of cnt = {train_df["cnt"].quantile(0.75):.0f}; '
      f'positive rate train {y_tr_g.mean():.3f}, val {y_vl_g.mean():.3f}')

COMMUTE = [7, 8, 9, 16, 17, 18, 19]
for nm, ytr_ in [('global', y_tr_g), ('grouped', y_tr)]:
    share = ytr_[train_df['hr'].isin(COMMUTE).values].sum() / ytr_.sum()
    print(f'  share of {nm:<8} positives that fall in the 7 commute hours {COMMUTE}: {share:.3f}   (7 of 24 hours = 0.292 if evenly spread)')

h_res = {}
for nm, ytr_, yvl_ in [('global', y_tr_g, y_vl_g), ('grouped', y_tr, y_vl)]:
    r = fit_logreg(H_tr, ytr_, max_iter=400000, tol=1e-9)   # the global-target hour-only fit is slow-converging; momentum makes it quick
    pv = lg.predict_proba(H_vl, r['w'], r['b'])
    h_res[nm] = dict(res=r, proba=pv, auc=lg.roc_auc(yvl_, pv))
    print(f'  hour-only model on the {nm:<8} target: val AUC = {h_res[nm]["auc"]:.4f}   (status {r["status"]}, {r["n_iter"]} iterations)')
AUC_HOUR_GLOBAL, AUC_HOUR_GROUPED = h_res['global']['auc'], h_res['grouped']['auc']

global-threshold target: train 75th percentile of cnt = 283; positive rate train 0.250, val 0.253
  share of global   positives that fall in the 7 commute hours [7, 8, 9, 16, 17, 18, 19]: 0.638   (7 of 24 hours = 0.292 if evenly spread)
  share of grouped  positives that fall in the 7 commute hours [7, 8, 9, 16, 17, 18, 19]: 0.298   (7 of 24 hours = 0.292 if evenly spread)


  hour-only model on the global   target: val AUC = 0.8500   (status converged, 4536 iterations)
  hour-only model on the grouped  target: val AUC = 0.4850   (status converged, 126 iterations)


## 5. Tests: gradient, step size, metrics (Req 6, 7, 10)

In [6]:
rng = np.random.default_rng(0)

# T1 gradient check: plain, L2, class-weighted; random and real data
ok_all = True
Xr = rng.standard_normal((60, 8)); yr = (rng.random(60) < 0.3).astype(float)
wr = rng.standard_normal(8) * 0.3; br = 0.2
for l2, sw in [(0.0, None), (0.1, None), (0.1, lg.balanced_weights(yr))]:
    g, l = lg.make_fns(l2, sw)
    gc = core.gradient_check(Xr, yr, wr, br, g, l)
    ok_all &= gc['all_pass']
    print(f'[{"PASS" if gc["all_pass"] else "FAIL"}] gradient check random data  l2={l2}  weighted={sw is not None}  '
          f'max_rel_w={gc["max_rel_error_w"]:.2e} max_rel_b={gc["max_rel_error_b"]:.2e}')
idx = rng.choice(len(X_tr), 2000, replace=False)
sw_real = lg.balanced_weights(y_tr[idx])
g, l = lg.make_fns(0.01, sw_real)
gc = core.gradient_check(X_tr[idx], y_tr[idx], rng.standard_normal(X_tr.shape[1]) * 0.05, 0.1, g, l, n_check=20)
ok_all &= gc['all_pass']
print(f'[{"PASS" if gc["all_pass"] else "FAIL"}] gradient check real data (2000 rows, l2=0.01, weighted)  '
      f'max_rel_w={gc["max_rel_error_w"]:.2e} max_rel_b={gc["max_rel_error_b"]:.2e}')
assert ok_all

# T2 stability: sigmoid / loss at extreme logits
z = np.array([-700.0, -30.0, 0.0, 30.0, 700.0])
assert np.isfinite(lg.sigmoid(z)).all()
Xe = np.array([[1000.0], [-1000.0]]); ye = np.array([0.0, 1.0])   # confidently wrong
assert np.isfinite(lg.bce_loss(Xe, ye, np.array([1.0]), 0.0))
print('[PASS] sigmoid and BCE finite at |z| up to 1000 (no overflow, no log(0))')

# T3 step size <= 1/L and monotone loss over 500 iterations
L = lg.lipschitz_logistic(X_tr)
g, l = lg.make_fns()
mono = core.gd_fit(X_tr, y_tr, lr=1.0 / L, max_iter=500, tol=0, gtol=0, grad_fn=g, loss_fn=l, record_every=1)
d = np.diff([v for _, v in mono['train_losses']])
assert bool(np.all(d <= 1e-12))
print(f'[PASS] L = {L:.4f}, step 1/L = {1/L:.4f}; training loss non-increasing over 500 iterations (max change {d.max():.2e})')

# T4 our numpy metrics vs sklearn on tied scores
ys = rng.integers(0, 2, 800); ss = np.round(rng.random(800) + 0.4 * ys, 2); ps = (ss > 0.8).astype(int)
assert abs(lg.roc_auc(ys, ss) - roc_auc_score(ys, ss)) < 1e-9
assert abs(lg.precision_recall_f1(ys, ps)[2] - f1_score(ys, ps)) < 1e-9
assert abs(lg.accuracy(ys, ps) - accuracy_score(ys, ps)) < 1e-12
print('[PASS] AUC / F1 / accuracy match sklearn on tied synthetic scores (< 1e-9)')

# T5 edge cases
try:
    lg.roc_auc(np.ones(5), np.arange(5.0)); raise SystemExit('no error raised')
except ValueError:
    print('[PASS] roc_auc raises ValueError on a single class')
assert lg.precision_recall_f1(np.array([1, 0]), np.array([0, 0])) == (0.0, 0.0, 0.0)
print('[PASS] precision / recall / F1 return 0.0 (no division error) when nothing is predicted positive')

[PASS] gradient check random data  l2=0.0  weighted=False  max_rel_w=5.61e-11 max_rel_b=7.32e-12
[PASS] gradient check random data  l2=0.1  weighted=False  max_rel_w=8.08e-11 max_rel_b=7.32e-12
[PASS] gradient check random data  l2=0.1  weighted=True  max_rel_w=1.79e-10 max_rel_b=4.84e-11
[PASS] gradient check real data (2000 rows, l2=0.01, weighted)  max_rel_w=3.36e-09 max_rel_b=1.44e-11
[PASS] sigmoid and BCE finite at |z| up to 1000 (no overflow, no log(0))


[PASS] L = 1.3044, step 1/L = 0.7666; training loss non-increasing over 500 iterations (max change -9.38e-06)
[PASS] AUC / F1 / accuracy match sklearn on tied synthetic scores (< 1e-9)
[PASS] roc_auc raises ValueError on a single class
[PASS] precision / recall / F1 return 0.0 (no division error) when nothing is predicted positive


## 6. Baselines (Req 11)

Two baselines on the grouped target, scored on validation: the **majority-class** predictor (always "not high") and the **hour-only** logistic model.

In [7]:
def show(name, ev, extra=''):
    print(f'{name:<34} acc {ev["accuracy"]:.4f}  prec {ev["precision"]:.4f}  rec {ev["recall"]:.4f}  F1 {ev["f1"]:.4f}  AUC {ev["auc"]:.4f}  {extra}')

BASE = {}
BASE['majority'] = lg.evaluate(y_vl, np.zeros(len(y_vl)), 0.5)
pv_h = h_res['grouped']['proba']
BASE['hour_only_0.5'] = lg.evaluate(y_vl, pv_h, 0.5)
bt_h = lg.best_threshold(y_vl, pv_h)
BASE['hour_only_tuned'] = lg.evaluate(y_vl, pv_h, bt_h['threshold'])
show('majority class (always 0)', BASE['majority'])
show('hour-only model, thr 0.5', BASE['hour_only_0.5'])
show('hour-only model, F1-tuned thr', BASE['hour_only_tuned'], f'(thr {bt_h["threshold"]:.2f})')

majority class (always 0)          acc 0.7498  prec 0.0000  rec 0.0000  F1 0.0000  AUC 0.5000  
hour-only model, thr 0.5           acc 0.7498  prec 0.0000  rec 0.0000  F1 0.0000  AUC 0.4850  
hour-only model, F1-tuned thr      acc 0.2718  prec 0.2530  rec 0.9780  F1 0.4020  AUC 0.4850  (thr 0.20)


## 7. Tuning: L2 strength x class weighting (Req 12)

All tuning is on the validation set, as the project rules state (so the validation numbers below are slightly optimistic). Each fit is our own gradient descent at step 1/L, warm-started along decreasing L2. Because class weights shift the probability scale, models are compared by **AUC** and by **F1 at each model's own best threshold**, not by F1 at a fixed 0.5.

In [8]:
L2_GRID = [1.0, 1e-1, 1e-2, 1e-3, 1e-4, 0.0]
tuning = []
t0 = time.time()
for cw in [False, True]:
    sw = lg.balanced_weights(y_tr) if cw else None
    w0, b0 = None, 0.0
    for l2 in L2_GRID:
        r = fit_logreg(X_tr, y_tr, l2=l2, sw=sw, w0=w0, b0=b0, X_val=X_vl, y_val=y_vl)
        w0, b0 = r['w'], r['b']
        pv = lg.predict_proba(X_vl, r['w'], r['b'])
        bt = lg.best_threshold(y_vl, pv)
        ev = lg.evaluate(y_vl, pv, 0.5)
        tuning.append(dict(l2=l2, class_weight=cw, status=r['status'], n_iter=r['n_iter'], auc=ev['auc'],
                           acc_at_05=ev['accuracy'], f1_at_05=ev['f1'], f1_best=bt['f1'], thr_best=bt['threshold'],
                           val_loss=r['final_val_loss'], w=r['w'], b=r['b']))
print(f'tuning fits took {time.time()-t0:.0f}s')
tdf = pd.DataFrame([{k: v for k, v in t.items() if k not in ('w', 'b')} for t in tuning])
print(tdf.to_string(index=False, float_format=lambda v: f'{v:.4f}'))

tuning fits took 19s
    l2  class_weight    status  n_iter    auc  acc_at_05  f1_at_05  f1_best  thr_best  val_loss
1.0000         False converged      38 0.8833     0.7502    0.0073   0.6795    0.2900    0.5113
0.1000         False converged      39 0.9194     0.8471    0.6263   0.7580    0.3400    0.3992
0.0100         False converged     110 0.9331     0.8747    0.7331   0.7704    0.3800    0.3110
0.0010         False converged     246 0.9359     0.8843    0.7591   0.7753    0.3700    0.2830
0.0001         False converged     365 0.9341     0.8857    0.7622   0.7742    0.4000    0.2826
0.0000         False converged    3195 0.9311     0.8861    0.7643   0.7743    0.3800    0.3052
1.0000          True converged      26 0.8906     0.7782    0.6548   0.6977    0.5400    0.6124
0.1000          True converged      40 0.9231     0.8421    0.7370   0.7613    0.5600    0.4689
0.0100          True converged     132 0.9342     0.8494    0.7457   0.7692    0.6600    0.3694
0.0010          Tru

### Selection rule

Choose, among settings whose best-threshold validation F1 is within 0.005 of the maximum, the **largest L2** and, on a tie, the **simpler** model (no class weights). The reasoning mirrors Phase 4: a difference below 0.005 F1 is within what a re-split of the 2,178 validation hours would change, so we prefer the more constrained model; class weights are only kept if they buy a clear improvement.

In [9]:
TOL_SEL = 0.005
best_f1 = max(t['f1_best'] for t in tuning)
cands = [t for t in tuning if t['f1_best'] >= best_f1 - TOL_SEL]
chosen = max(cands, key=lambda t: (t['l2'], not t['class_weight']))
print(f'best F1 over the table: {best_f1:.4f};  {len(cands)} settings within {TOL_SEL}')
print(f'CHOSEN: l2 = {chosen["l2"]}, class_weight = {chosen["class_weight"]}  '
      f'(AUC {chosen["auc"]:.4f}, F1 at own best threshold {chosen["f1_best"]:.4f}, status {chosen["status"]})')
print('statuses in table:', sorted({t['status'] for t in tuning}))

best F1 over the table: 0.7753;  4 settings within 0.005
CHOSEN: l2 = 0.01, class_weight = False  (AUC 0.9331, F1 at own best threshold 0.7704, status converged)
statuses in table: ['converged']


## 8. Final fit and decision threshold (Req 8, 13)

In [10]:
L2_F, CW_F = chosen['l2'], chosen['class_weight']
SW_F = lg.balanced_weights(y_tr) if CW_F else None

t0 = time.time()
final = fit_logreg(X_tr, y_tr, l2=L2_F, sw=SW_F, X_val=X_vl, y_val=y_vl)      # cold start from zeros
W_F, B_F = final['w'], final['b']
print(f'final fit: status {final["status"]}, {final["n_iter"]} iterations, step {final["lr"]:.4f} (= 1/L, L = {final["L"]:.4f}), {time.time()-t0:.0f}s')
print(f'final train loss {final["final_train_loss"]:.6f}   val loss {final["final_val_loss"]:.6f}')
assert final['status'] == 'converged', final['status']

fig, ax = plt.subplots(figsize=(7, 4))
it_tr, l_tr = zip(*final['train_losses']); it_vl, l_vl = zip(*final['val_losses'])
ax.semilogx(it_tr, l_tr, label='train'); ax.semilogx(it_vl, l_vl, label='validation (unweighted)')
ax.set_xlabel('iteration'); ax.set_ylabel('cross-entropy loss'); ax.set_title('Phase 5 convergence'); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(PLOTS / '05_convergence.png', dpi=120); plt.close(fig)
print('[SAVED] plots/05_convergence.png')

p_vl = lg.predict_proba(X_vl, W_F, B_F)
bt = lg.best_threshold(y_vl, p_vl)
THR_F = bt['threshold']
print(f'decision threshold tuned on validation F1 over {len(bt["grid"])} values: {THR_F:.2f} (F1 {bt["f1"]:.4f}); default 0.5')
print('NOTE: the threshold is tuned and reported on the same validation set, so the tuned-threshold numbers are optimistic.')

fig, ax = plt.subplots(figsize=(7, 4))
P_ = [lg.precision_recall_f1(y_vl, (p_vl >= t).astype(int)) for t in bt['grid']]
ax.plot(bt['grid'], [p[0] for p in P_], label='precision'); ax.plot(bt['grid'], [p[1] for p in P_], label='recall')
ax.plot(bt['grid'], bt['f1_grid'], label='F1', lw=2); ax.axvline(THR_F, color='k', ls=':', label=f'chosen {THR_F:.2f}')
ax.set_xlabel('probability threshold'); ax.set_title('Threshold sweep (validation)'); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(PLOTS / '05_threshold_sweep.png', dpi=120); plt.close(fig)
print('[SAVED] plots/05_threshold_sweep.png')

final fit: status converged, 105 iterations, step 0.7608 (= 1/L, L = 1.3144), 0s
final train loss 0.298199   val loss 0.310984
[SAVED] plots/05_convergence.png
decision threshold tuned on validation F1 over 99 values: 0.38 (F1 0.7704); default 0.5
NOTE: the threshold is tuned and reported on the same validation set, so the tuned-threshold numbers are optimistic.


[SAVED] plots/05_threshold_sweep.png


## 9. Final evaluation on the validation set (Req 14)

In [11]:
EV05 = lg.evaluate(y_vl, p_vl, 0.5)
EVT = lg.evaluate(y_vl, p_vl, THR_F)
show('FINAL model, thr 0.5', EV05)
show('FINAL model, tuned thr', EVT, f'(thr {THR_F:.2f})')
print('\nconfusion matrix at tuned threshold  [[TN, FP], [FN, TP]]:')
print(np.array(EVT['confusion']))
print(f'positive rate: true {EVT["positive_rate_true"]:.4f}, predicted {EVT["positive_rate_pred"]:.4f}')

# our numpy metrics vs sklearn on the real validation predictions (Req 10)
pred_t = (p_vl >= THR_F).astype(int)
assert abs(EVT['auc'] - roc_auc_score(y_vl, p_vl)) < 1e-9
assert abs(EVT['f1'] - f1_score(y_vl, pred_t)) < 1e-9
assert abs(EVT['accuracy'] - accuracy_score(y_vl, pred_t)) < 1e-12
print('[PASS] validation AUC / F1 / accuracy equal sklearn.metrics to < 1e-9')

print('\nvs baselines (F1 / AUC):')
print(f'  majority          F1 {BASE["majority"]["f1"]:.4f}  AUC {BASE["majority"]["auc"]:.4f}')
print(f'  hour-only (tuned) F1 {BASE["hour_only_tuned"]["f1"]:.4f}  AUC {BASE["hour_only_tuned"]["auc"]:.4f}')
print(f'  final   (tuned)   F1 {EVT["f1"]:.4f}  AUC {EVT["auc"]:.4f}')
BEATS = (EVT['f1'] > BASE['majority']['f1'] and EVT['f1'] > BASE['hour_only_tuned']['f1']
         and EVT['auc'] > BASE['majority']['auc'] and EVT['auc'] > BASE['hour_only_tuned']['auc'])
print('final model beats both baselines on F1 and AUC:', BEATS)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for nm, pv_, c in [('final model', p_vl, 'C0'), ('hour-only (grouped target)', h_res['grouped']['proba'], 'C1')]:
    fpr, tpr = lg.roc_curve(y_vl, pv_); axes[0].plot(fpr, tpr, c, label=f'{nm} (AUC {lg.roc_auc(y_vl, pv_):.3f})')
axes[0].plot([0, 1], [0, 1], 'k:'); axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR'); axes[0].set_title('ROC (validation)'); axes[0].legend(); axes[0].grid(alpha=0.3)
grid = np.linspace(0.01, 0.99, 99)
for nm, pv_, c in [('final model', p_vl, 'C0'), ('hour-only (grouped target)', h_res['grouped']['proba'], 'C1')]:
    pr = np.array([lg.precision_recall_f1(y_vl, (pv_ >= t).astype(int))[:2] for t in grid])
    axes[1].plot(pr[:, 1], pr[:, 0], c, label=nm)
axes[1].axhline(y_vl.mean(), color='k', ls=':', label='positive rate'); axes[1].set_xlabel('recall'); axes[1].set_ylabel('precision')
axes[1].set_title('Precision-recall (validation)'); axes[1].legend(); axes[1].grid(alpha=0.3)
fig.tight_layout(); fig.savefig(PLOTS / '05_roc_pr.png', dpi=120); plt.close(fig)
print('[SAVED] plots/05_roc_pr.png')

FINAL model, thr 0.5               acc 0.8747  prec 0.7845  rec 0.6881  F1 0.7331  AUC 0.9331  
FINAL model, tuned thr             acc 0.8788  prec 0.7322  rec 0.8128  F1 0.7704  AUC 0.9331  (thr 0.38)

confusion matrix at tuned threshold  [[TN, FP], [FN, TP]]:
[[1471  162]
 [ 102  443]]
positive rate: true 0.2502, predicted 0.2778
[PASS] validation AUC / F1 / accuracy equal sklearn.metrics to < 1e-9

vs baselines (F1 / AUC):
  majority          F1 0.0000  AUC 0.5000
  hour-only (tuned) F1 0.4020  AUC 0.4850
  final   (tuned)   F1 0.7704  AUC 0.9331
final model beats both baselines on F1 and AUC: True
[SAVED] plots/05_roc_pr.png


## 10. Chronological check (Req 15)

Same cut as Phase 3: sort by `dteday`, `hr`; the first 80% trains, the last 20% validates. Everything is refit on the earlier part only: group thresholds, preprocessing parameters, polynomial parameters, and the classifier (same L2 and class-weight setting, from zeros). The decision threshold is the one tuned on the seeded split (not re-tuned).

In [12]:
chrono_all = train_all.sort_values(['dteday', 'hr']).reset_index(drop=True)
cut = int(len(chrono_all) * 0.8)
c_tr, c_vl = chrono_all.iloc[:cut], chrono_all.iloc[cut:]
print(f'chronological cut after {c_tr["dteday"].max()}  |  train {len(c_tr)}  val {len(c_vl)}')

pm_c = core.fit_preprocessor(c_tr)
X1c, nc1 = core.transform(c_tr, pm_c)
pp_c = p3.fit_poly(X1c, nc1, meta_p3['complexity_spec'])
Xc_tr, _ = lg.build_features(c_tr, pm_c, pp_c, FEATS)
Xc_vl, _ = lg.build_features(c_vl, pm_c, pp_c, FEATS)
THR_C = lg.group_thresholds(c_tr)
yc_tr, yc_vl = lg.make_target(c_tr, THR_C), lg.make_target(c_vl, THR_C)
print(f'positive rate: chrono-train {yc_tr.mean():.3f}   chrono-val {yc_vl.mean():.3f}   (seeded val {y_vl.mean():.3f})')

SWc = lg.balanced_weights(yc_tr) if CW_F else None
rc = fit_logreg(Xc_tr, yc_tr, l2=L2_F, sw=SWc)
pc = lg.predict_proba(Xc_vl, rc['w'], rc['b'])
CH05, CHT = lg.evaluate(yc_vl, pc, 0.5), lg.evaluate(yc_vl, pc, THR_F)
CH_MAJ = lg.evaluate(yc_vl, np.zeros(len(yc_vl)), 0.5)
print(f'chronological fit status: {rc["status"]} ({rc["n_iter"]} iterations)\n')
show('seeded split, tuned thr', EVT, f'(thr {THR_F:.2f})')
show('chronological, same thr', CHT, f'(thr {THR_F:.2f})')
show('chronological, thr 0.5', CH05)
show('chronological, majority class', CH_MAJ)

chronological cut after 2012-08-05  |  train 8708  val 2178


positive rate: chrono-train 0.245   chrono-val 0.640   (seeded val 0.250)


chronological fit status: converged (114 iterations)

seeded split, tuned thr            acc 0.8788  prec 0.7322  rec 0.8128  F1 0.7704  AUC 0.9331  (thr 0.38)
chronological, same thr            acc 0.7461  prec 0.8052  rec 0.7954  F1 0.8003  AUC 0.8125  (thr 0.38)
chronological, thr 0.5             acc 0.7410  prec 0.8463  rec 0.7272  F1 0.7822  AUC 0.8125  
chronological, majority class      acc 0.3604  prec 0.0000  rec 0.0000  F1 0.0000  AUC 0.5000  


## 11. Determinism and artifacts (Req 16-17)

In [13]:
final2 = fit_logreg(X_tr, y_tr, l2=L2_F, sw=SW_F)
assert np.array_equal(W_F, final2['w']) and B_F == final2['b']
W_SHA = hashlib.sha256(W_F.tobytes()).hexdigest()
print(f'[PASS] two cold-start fits give bit-identical weights; SHA-256(w) = {W_SHA}')

meta5 = {
    'seed': SEED, 'ids': sorted(IDS),
    'target': 'high = cnt > train 0.75-quantile of cnt within (hr, workingday)',
    'quantile': lg.HIGH_QUANTILE,
    'group_thresholds': lg.thresholds_to_json(THR),
    'feature_names': FEATS, 'n_features': len(FEATS),
    'phase4_l1_w_sha256': meta_p4['phase4_l1_w_sha256'], 'phase3_w_sha256': meta_p3['phase3_w_sha256'],
    'l2': L2_F, 'class_weight': CW_F, 'lr': final['lr'], 'lipschitz': final['L'],
    'momentum': True, 'max_iter': FIT_KW['max_iter'], 'tol': FIT_KW['tol'], 'gtol': FIT_KW['gtol'],
    'iterations_used': final['n_iter'], 'status': final['status'],
    'final_train_loss': final['final_train_loss'], 'final_val_loss': final['final_val_loss'],
    'decision_threshold': THR_F, 'selection_tolerance': TOL_SEL,
    'val_metrics_tuned_threshold': EVT, 'val_metrics_threshold_0.5': EV05,
    'baselines': {k: v for k, v in BASE.items()},
    'hour_only_auc_global_target': AUC_HOUR_GLOBAL, 'hour_only_auc_grouped_target': AUC_HOUR_GROUPED,
    'chrono_cut_date': str(c_tr['dteday'].max()),
    'chrono_metrics_same_threshold': CHT, 'chrono_metrics_threshold_0.5': CH05,
    'tuning_table': [{k: v for k, v in t.items() if k not in ('w', 'b')} for t in tuning],
}
sha_saved = lg.save_artifacts(P5_STEM, W_F, B_F, meta5, train_idx, val_idx)
assert sha_saved == W_SHA

# Round trip: reload and reproduce the validation AUC
w_rt, b_rt, meta_rt, _, _ = lg.load_artifacts(P5_STEM)
auc_rt = lg.roc_auc(y_vl, lg.predict_proba(X_vl, w_rt, b_rt))
assert abs(auc_rt - meta_rt['val_metrics_tuned_threshold']['auc']) < 1e-12
assert hashlib.sha256(w_rt.tobytes()).hexdigest() == meta_rt['w_sha256']
print(f'[PASS] artifacts saved and reloaded; validation AUC reproduced ({auc_rt:.12f})')

[PASS] two cold-start fits give bit-identical weights; SHA-256(w) = 8261ee55935a8217090a0637a542efb743cf160c5287570f449c28ef5d530c62
[PASS] artifacts saved and reloaded; validation AUC reproduced (0.933074152935)


## 12. No-sklearn check (Req 9)

In [14]:
# NO-SKLEARN-CHECK  (this cell is excluded from the search)
pat = re.compile(r'(?<!core)\.fit\(|(?<!core)\.predict\(|LogisticRegression')   # core.predict is our own numpy predictor
nb = json.load(open('logistic_regression.ipynb', encoding='utf-8'))
hits = []
for i, c in enumerate(nb['cells']):
    s = ''.join(c['source'])
    if c['cell_type'] == 'code' and '# NO-SKLEARN-CHECK' not in s and pat.search(s):
        hits.append(('notebook cell', i))
if pat.search(Path('logistic_regression.py').read_text(encoding='utf-8')):
    hits.append(('logistic_regression.py', None))
assert not hits, hits
print('[PASS] no .fit( / .predict( / LogisticRegression in the module or in any other code cell')

[PASS] no .fit( / .predict( / LogisticRegression in the module or in any other code cell


## 13. Pipeline retrospective (Req 19)

All numbers below are read from the artifacts or recomputed from them in this run.

In [15]:
yv = val_df['cnt'].values.astype(float)
w1, b1, _, _, _ = core.load_artifacts(P1_STEM)
p1_pred = core.predict(X1_vl, w1, b1)
_, _, m2 = p2.load_artifacts(P2_STEM)
X_p3_vl, _ = lg.build_features(val_df, meta_p1, poly_params, meta_p3['expanded_feature_names'])
p3_pred = core.predict(X_p3_vl, w_p3, b_p3)

retro = pd.DataFrame([
    ['1 Gradient descent',   'linear',                            len(w1),   core.r2_score_np(yv, p1_pred), core.rmse_np(yv, p1_pred), np.nan],
    ['2 Polynomial (d=3)',   'powers of temp/hum/wind/days',      m2['n_new_cols'] + len(w1), m2['val_r2'], m2['val_rmse'], np.nan],
    ['3 Bias-variance',      meta_p3['chosen_rung'][:34],         len(w_p3), core.r2_score_np(yv, p3_pred), core.rmse_np(yv, p3_pred), meta_p3['chrono_r2_gd']],
    ['4 Lasso',              'same, L1 (lambda %.3g)' % meta_p4['l1_lambda'], len(FEATS), meta_p4['l1_r2_val'], meta_p4['l1_rmse_val'], meta_p4['chrono_r2_lasso']],
], columns=['phase', 'model', 'n_features', 'val_R2', 'val_RMSE', 'chrono_R2'])
print(retro.to_string(index=False, float_format=lambda v: f'{v:.4f}'))

print('\nPhase 5 (classification, grouped target):')
retro5 = pd.DataFrame([
    ['majority class',               BASE['majority']['accuracy'],        BASE['majority']['f1'],        BASE['majority']['auc'],        np.nan],
    ['hour-only logistic (tuned thr)', BASE['hour_only_tuned']['accuracy'], BASE['hour_only_tuned']['f1'], BASE['hour_only_tuned']['auc'], np.nan],
    ['5 Logistic, %d features (tuned thr)' % len(FEATS), EVT['accuracy'], EVT['f1'], EVT['auc'], CHT['auc']],
], columns=['model', 'val_accuracy', 'val_F1', 'val_AUC', 'chrono_AUC'])
print(retro5.to_string(index=False, float_format=lambda v: f'{v:.4f}'))

             phase                        model  n_features  val_R2  val_RMSE  chrono_R2
1 Gradient descent                       linear          54  0.6781  100.4427        NaN
2 Polynomial (d=3) powers of temp/hum/wind/days          62  0.6858   99.2354        NaN
   3 Bias-variance                 R9 + hr x yr         589  0.9104   53.0038     0.9063
           4 Lasso      same, L1 (lambda 0.251)         470  0.9079   53.7160     0.8938

Phase 5 (classification, grouped target):
                               model  val_accuracy  val_F1  val_AUC  chrono_AUC
                      majority class        0.7498  0.0000   0.5000         NaN
      hour-only logistic (tuned thr)        0.2718  0.4020   0.4850         NaN
5 Logistic, 470 features (tuned thr)        0.8788  0.7704   0.9331      0.8125


### Discussion

In [16]:
# Every number quoted in the Outcome below is printed here (Req 18)
print(f'seed {SEED}; train/val rows {len(train_df)}/{len(val_df)}; features {len(FEATS)}')
print(f'positive rate: train {y_tr.mean():.4f}, val {y_vl.mean():.4f}')
print(f'hour-only AUC: global target {AUC_HOUR_GLOBAL:.4f}, grouped target {AUC_HOUR_GROUPED:.4f}')
print(f'chosen: l2 {L2_F}, class_weight {CW_F}; status {final["status"]}, iterations {final["n_iter"]}, decision threshold {THR_F:.2f}')
print(f'FINAL val @0.5:  acc {EV05["accuracy"]:.4f} F1 {EV05["f1"]:.4f} AUC {EV05["auc"]:.4f}')
print(f'FINAL val @thr:  acc {EVT["accuracy"]:.4f} prec {EVT["precision"]:.4f} rec {EVT["recall"]:.4f} F1 {EVT["f1"]:.4f} AUC {EVT["auc"]:.4f}')
print(f'majority: acc {BASE["majority"]["accuracy"]:.4f} F1 {BASE["majority"]["f1"]:.4f} AUC {BASE["majority"]["auc"]:.4f}')
print(f'hour-only tuned: acc {BASE["hour_only_tuned"]["accuracy"]:.4f} F1 {BASE["hour_only_tuned"]["f1"]:.4f} AUC {BASE["hour_only_tuned"]["auc"]:.4f}')
print(f'chrono @same thr: acc {CHT["accuracy"]:.4f} F1 {CHT["f1"]:.4f} AUC {CHT["auc"]:.4f}; chrono positive rate {yc_vl.mean():.4f}')
print(f'tuning: AUC range {min(t["auc"] for t in tuning):.4f}..{max(t["auc"] for t in tuning):.4f}; F1(best thr) range {min(t["f1_best"] for t in tuning):.4f}..{max(t["f1_best"] for t in tuning):.4f}')
print(f'final model beats both baselines: {BEATS}')
print(f'total runtime {time.time()-T0:.0f}s')

seed 42648; train/val rows 8708/2178; features 470
positive rate: train 0.2441, val 0.2502
hour-only AUC: global target 0.8500, grouped target 0.4850
chosen: l2 0.01, class_weight False; status converged, iterations 105, decision threshold 0.38
FINAL val @0.5:  acc 0.8747 F1 0.7331 AUC 0.9331
FINAL val @thr:  acc 0.8788 prec 0.7322 rec 0.8128 F1 0.7704 AUC 0.9331
majority: acc 0.7498 F1 0.0000 AUC 0.5000
hour-only tuned: acc 0.2718 F1 0.4020 AUC 0.4850
chrono @same thr: acc 0.7461 F1 0.8003 AUC 0.8125; chrono positive rate 0.6396
tuning: AUC range 0.8833..0.9359; F1(best thr) range 0.6795..0.7753
final model beats both baselines: True
total runtime 24s


## Outcome